In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print("✅ GPU detected!\n")
    print(result.stdout)
else:
    print("❌ No GPU! Go to Runtime → Change runtime type → T4 GPU")


✅ GPU detected!

Wed Apr  1 10:56:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   71C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+------------------------------

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device.upper()}")
if device == "cuda":
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

print(f"\n📥 Downloading {MODEL_NAME} (~2 GB, please wait)...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.eval()
print("\n✅ Model ready!")

🖥️  Device: CUDA
   GPU : Tesla T4
   VRAM: 15.6 GB

📥 Downloading Qwen/Qwen2.5-3B-Instruct (~2 GB, please wait)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


✅ Model ready!


In [ ]:
conversation_history = []

def chat(user_input, max_new_tokens=512):
    global conversation_history

    messages = [
        {"role": "system", "content": "You are a helpful, knowledgeable assistant. Answer accurately and concisely."}
    ]
    for user_msg, bot_msg in conversation_history[-6:]:
        messages.append({"role": "user",      "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": user_input})

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(
        output[0][inputs.input_ids.shape[-1]:],
        skip_special_tokens=True
    ).strip()

    conversation_history.append((user_input, response))
    return response

print("✅ Chat function ready.")

✅ Chat function ready.


In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

turn = 0
conversation_log = []

chat_output = widgets.Output()
user_input  = widgets.Text(
    placeholder="Ask anything…",
    layout=widgets.Layout(width="75%", height="38px")
)
send_btn  = widgets.Button(description="Send 💬",  button_style="primary",
                           layout=widgets.Layout(width="12%", height="38px"))
reset_btn = widgets.Button(description="Reset 🔄", button_style="warning",
                           layout=widgets.Layout(width="11%", height="38px"))
status = widgets.Label(value=f"✅ Qwen2.5-3B ready on {device.upper()}")

print("✅ UI built.")

✅ UI built.


In [ ]:
def render():
    with chat_output:
        clear_output(wait=True)
        html = '<div style="background:#1e1e2e;border-radius:10px;padding:14px;height:350px;overflow-y:auto;font-family:monospace;font-size:14px;">'
        for role, text in conversation_log:
            if role == "user":
                html += f'<div style="color:#89dceb;margin:8px 0;"><b>You:</b> {text}</div>'
            elif role == "bot":
                html += f'<div style="color:#a6e3a1;margin:8px 0;"><b>Qwen:</b> {text}</div>'
            else:
                html += f'<div style="color:#f9e2af;margin:4px 0;font-style:italic;">{text}</div>'
        html += '</div>'
        display(HTML(html))

def on_send(_):
    global turn
    text = user_input.value.strip()
    if not text:
        return
    user_input.value = ""
    conversation_log.append(("user", text))
    render()
    status.value = "⏳ Thinking…"
    send_btn.disabled = True
    try:
        response = chat(text)
        turn += 1
        vram = ""
        if device == "cuda":
            used  = torch.cuda.memory_allocated() / 1e9
            total = torch.cuda.get_device_properties(0).total_memory / 1e9
            vram  = f"  |  VRAM {used:.1f}/{total:.1f} GB"
        conversation_log.append(("bot", response))
        status.value = f"✅ Turn #{turn}{vram}"
    except Exception as e:
        conversation_log.append(("system", f"❌ Error: {e}"))
        status.value = "❌ Error"
    finally:
        send_btn.disabled = False
        render()

def on_reset(_):
    global turn
    turn = 0
    conversation_history.clear()
    conversation_log.clear()
    conversation_log.append(("system", "🔄 Reset! Start fresh."))
    status.value = f"✅ Ready"
    render()

send_btn.on_click(on_send)
reset_btn.on_click(on_reset)
user_input.on_submit(on_send)

print("✅ Logic wired.")

✅ Logic wired.


In [ ]:
conversation_log.append(("system", f"🤖 Qwen2.5-3B on {device.upper()} — ask me anything!"))
render()
display(widgets.VBox([
    chat_output,
    widgets.HBox([user_input, send_btn, reset_btn]),
    status
]))